# Multi-model ERP experiments — Argus integration (Notebook 09)

One settings cell, one kernel, sequential model processes. Run the numbered steps in order. All model processes use your existing **carla360-models** environment; no packages or weights are installed here.

**GT evaluation and GT conditioning are disabled.** Direct ERP inference for VGGT, VGGT-Omega and OmniVGGT is an experimental use of perspective models. Their native depth is not the same quantity as panoramic radial depth. Successful inference and finite outputs do not establish geometric accuracy.

| Model | Integration evidence | New wrapper status |
|---|---|---|
| VGGT | User supplied successful inference code | GPU validation on Rails still required |
| VGGT-Omega | User supplied successful inference code | GPU validation on Rails still required |
| OmniVGGT | Successful direct-ERP code and output log supplied | GPU validation on Rails still required |
| PanoVGGT | Official preprocessing/architecture; successful official demo reported | Raw-array export and strict loading need Rails validation |
| VGGT-360 | Official projection/fusion reused; earlier single-image run reported | Local VGGT checkpoint compatibility and this selection untested |
| Argus | Checkpoint access pending | Disabled, no substitute weights |

This new notebook does not change notebooks 06/07 or previous output folders. Each model runs in its own process to isolate identically named modules and release GPU memory when it exits.

Argus: three-image smoke test passed on H100 with 504x252 ERP inputs; the integrated worker has **not** yet been tested on Rails. Checkpoint loads with strict=False and unused point-head unexpected keys. Metric accuracy and equivalence to official demo preprocessing remain unverified.


## Step 1 — Settings (edit only this cell)

`START`, `STOP` and `STRIDE` refer to positions in the sorted file list, **not IDs embedded in filenames**. STOP is exclusive. Use `ORDER="lexical"` to reproduce the previous `sorted(glob(...))` test; natural order sorts numeric filename parts numerically for longer sequences.

For more data: set INPUT_DIR to `ROOT / "datasets/images"`, set LIMIT to a larger number or None, and keep CHUNK_SIZE small. CHUNK_SIZE is the number of frames jointly processed by a model. Independent chunks have independent pose origins/scales; they are never stitched into one trajectory. Increasing CHUNK_SIZE changes context and memory usage, not just speed. Use a new RUN_NAME when changing image selection or chunk boundaries.


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "comparison" / "bench_ui.py").is_file(), "Open this notebook inside carla360-benchmark."
sys.path.insert(0, str(ROOT))

PYTHON = Path.home() / ".conda/envs/carla360-models/bin/python"
INPUT_DIR = ROOT / "datasets/test_001"
IMAGE_PATTERN = "*.png"
ORDER = "lexical"       # lexical: previous three-image test; natural: numeric filename order
START = 0
STOP = None
STRIDE = 1
LIMIT = None             # None selects all matching images; check the preview before running
CHUNK_SIZE = 3           # Joint temporal context per inference call; 1 is single-image inference
RUN_NAME = "erp_models_argus_001"
SELECTED_MODELS = ["vggt", "vggt_omega", "panovggt", "omnivggt", "vggt360", "argus"]
ACTION = "preview"      # preview | run | view
RERUN = False            # True archives previous results before recomputing
CONTINUE_ON_ERROR = True # Keep other model/chunk results if one fails
DEVICE = "cuda"
PRECISION = "bf16"       # bf16 reproduces supplied smoke tests; fp16/fp32 are untested alternatives

MODEL_SETTINGS = {
    "vggt": {"repo": str(ROOT / "third_party/VGGT"), "checkpoint": str(ROOT / "checkpoints/VGGT/model.pt")},
    "vggt_omega": {"repo": str(ROOT / "third_party/VGGT-Omega"), "checkpoint": str(ROOT / "checkpoints/VGGT-Omega/vggt_omega_1b_416_reproduce.pt"), "resolution": 416},
    "panovggt": {"repo": str(ROOT / "third_party/PanoVGGT"), "checkpoint": str(ROOT / "checkpoints/panovggt.pt"), "config": str(ROOT / "third_party/PanoVGGT/training/config/default.yaml")},
    "omnivggt": {"repo": str(ROOT / "third_party/OmniVGGT"), "checkpoint": str(ROOT / "checkpoints/OmniVGGT/OmniVGGT.safetensors"), "resolution": 518},
    "vggt360": {"repo": str(ROOT / "third_party/VGGT-360"), "checkpoint": str(ROOT / "checkpoints/VGGT/model.pt"), "fov": 110},
    "argus": {"repo": str(ROOT / "third_party/Argus"), "checkpoint": str(ROOT / "checkpoints/Argus/argus_realsee3d.pt"), "resolution": 504},
}
PLOT_CHUNK = 0
PLOT_FRAME = 0           # Position within PLOT_CHUNK
MAX_POINTS = 10000      # Display subsampling only; raw saved points are preserved


## Step 2 — Preview selected images and execution size

Nothing runs on the GPU in this step. Check the first/last filenames and selected image count. A very large selection creates many independent jobs, rather than loading the full dataset onto the GPU. Staging copies selected files without changing their bytes.


In [ ]:
import json
import math
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display
from comparison import bench_ui as bench

assert ACTION in ("preview", "run", "view")
assert RUN_NAME and Path(RUN_NAME).name == RUN_NAME and RUN_NAME not in (".", "..")
paths = bench.select_images(INPUT_DIR, IMAGE_PATTERN, START, STOP, STRIDE, LIMIT, ORDER)
RUN_DIR = ROOT / "outputs/notebook_benchmark" / RUN_NAME
plan = bench.make_plan(paths, RUN_DIR, MODEL_SETTINGS, SELECTED_MODELS, CHUNK_SIZE, PRECISION, DEVICE)
print("Selected images:", len(paths), "| Chunks:", math.ceil(len(paths) / CHUNK_SIZE))
print("Selected models:", SELECTED_MODELS, "| Action:", ACTION)
print("Output:", RUN_DIR)
print("GT input/evaluation: disabled. No cross-chunk pose alignment.")
display(pd.DataFrame(plan["image_records"]).iloc[list(dict.fromkeys([*range(min(5, len(paths))), len(paths)-1]))])
fig, axes = plt.subplots(1, min(3, len(paths)), figsize=(12, 3), squeeze=False)
for ax, path in zip(axes[0], paths[:3]):
    ax.imshow(Image.open(path)); ax.set_title(path.name, fontsize=7); ax.axis("off")
plt.show()


## Step 3 — Environment and model checks

In run mode, check the shared Python, CUDA, import paths and local files. No model weights are loaded yet. Strict checkpoint compatibility is tested during inference; failures stop that job instead of silently accepting missing keys. VGGT-360 may apply the existing narrowly checked fix for its unused missing attention import. Preview/view mode skips GPU checks.


In [ ]:
if ACTION == "run":
    preflight_rows = bench.preflight(plan, PYTHON)
    display(pd.DataFrame(preflight_rows))
else:
    print("GPU checks skipped. Set ACTION='run' in Step 1 when the preview is correct.")


## Step 4 — Run selected models sequentially, or reuse completed chunks

A separate process runs each model/chunk using PYTHON above. Outputs contain raw Float32 arrays, image hashes, checkpoint hash, source revision, settings, shape/finite checks, timing and GPU usage. Metadata is written only after success. Completed results with matching inputs/settings/weights/code are reused. Failed or explicitly rerun attempts are archived. Errors and logs remain available; no automatic parameter changes or OOM retries occur.

If memory runs out, choose a smaller CHUNK_SIZE and a **new RUN_NAME**. Model load time is included in timing, and weights are reloaded for each chunk. This conservative implementation favors reproducibility and memory isolation over maximum throughput.


In [ ]:
if ACTION == "run":
    ready = [row["model"] for row in preflight_rows if row.get("ready")]
    if not ready:
        raise RuntimeError("No model passed Step 3. Resolve its reported error first.")
    skipped = [row for row in preflight_rows if not row.get("ready")]
    run_status = bench.execute({**plan, "selected": ready}, PYTHON, RERUN, CONTINUE_ON_ERROR) + skipped
    display(pd.DataFrame(run_status))
    (RUN_DIR / "last_run_status.json").write_text(json.dumps(run_status, indent=2))
elif ACTION == "view":
    if not (RUN_DIR / "manifest.json").exists():
        raise FileNotFoundError("No saved run. Select an existing RUN_NAME or use ACTION='run'.")
    print("Viewing saved results; no inference.")
else:
    print("Preview only: no model was run. Change ACTION to 'run' and rerun Steps 1–4.")


## Step 5 — Compare native depth maps for one selected frame

Change PLOT_CHUNK/PLOT_FRAME in Step 1 to inspect any part of the dataset. Each panel retains its native resolution; the input RGB is a reference, not a guarantee of identical pixel coverage after upstream preprocessing. Normalized plots use independent 2nd/98th percentiles and hide scale. Raw plots show native head values: they are **not common metric depth**, and z-depth on direct ERP is not converted into spherical range. Blank panels mean no completed result.


In [ ]:
if ACTION != "preview":
    for normalized in (True, False):
        fig = bench.depth_plot(plan, PLOT_CHUNK, PLOT_FRAME, normalized)
        fig.savefig(RUN_DIR / f"depth_chunk{PLOT_CHUNK:05d}_frame{PLOT_FRAME:04d}_{'normalized' if normalized else 'raw'}.png", dpi=150, bbox_inches="tight")
        plt.show()
else:
    print("Plots are available after a run or in view mode.")


## Step 6 — Sanity checks, per-model poses and native point clouds

Shows each model independently in its own coordinate system. No GT metrics, scale alignment, pose ranking or cross-chunk trajectory stitching. Rotation checks are algebraic sanity checks, not proof of physical correctness. Point clouds are displayed only when a native world-point output exists; no pinhole unprojection is applied to ERP depth. VGGT-360 has no temporal trajectory in this wrapper.


In [ ]:
if ACTION != "preview":
    figures, checks = bench.geometry_plots(plan, PLOT_CHUNK, MAX_POINTS)
    display(pd.DataFrame(checks))
    (RUN_DIR / f"sanity_chunk{PLOT_CHUNK:05d}.json").write_text(json.dumps(checks, indent=2))
    for i, fig in enumerate(figures):
        fig.savefig(RUN_DIR / f"geometry_chunk{PLOT_CHUNK:05d}_{i:02d}.png", dpi=120, bbox_inches="tight")
        plt.show()


## Step 7 — Inspect earlier manual outputs without overwriting them

Your previous outputs under outputs/<model>/test_001 are preserved. This table inventories their raw depth and metadata. Old files without pixel hashes/configuration cannot be silently treated as verified cached results for this new run. PanoVGGT PNG depth previews are not raw Float32 predictions and are not imported as numerical depth. The new run creates its own verified cache.


In [ ]:
import numpy as np
legacy_rows = []
for name in SELECTED_MODELS:
    folder = ROOT / "outputs" / name / "test_001"
    row = {"model": name, "folder_exists": folder.is_dir(), "path": str(folder)}
    if (folder / "depth.npy").is_file():
        try:
            depth = np.load(folder / "depth.npy", mmap_mode="r", allow_pickle=False)
            row["raw_depth_shape"] = str(depth.shape)
        except Exception as exc:
            row["error"] = str(exc)
    row["metadata_present"] = (folder / "metadata.json").is_file()
    legacy_rows.append(row)
display(pd.DataFrame(legacy_rows))


## Step 8 — Export a small debugging/report ZIP

Includes metadata, settings, logs and plots. Excludes raw images, predictions and checkpoints. The full raw predictions remain in each chunk/model folder. The report records paths on your server.


In [ ]:
from datetime import datetime, timezone
import zipfile
if RUN_DIR.exists() and ACTION != "preview":
    report = RUN_DIR / f"report_{datetime.now(timezone.utc):%Y%m%dT%H%M%S_%fZ}.zip"
    with zipfile.ZipFile(report, "x", zipfile.ZIP_DEFLATED) as archive:
        for path in RUN_DIR.rglob("*"):
            if path.is_file() and "archived" not in path.relative_to(RUN_DIR).parts and "images" not in path.relative_to(RUN_DIR).parts and path.suffix in (".json", ".log", ".png"):
                archive.write(path, path.relative_to(RUN_DIR))
    print("Report:", report)
